# 1. Data Understanding 

In [1]:
import pandas as pd

df = pd.read_csv("cleaned_reviews.csv")
df.head()

,sentiments,cleaned_review,cleaned_review_length,review_score
0,positive,i wish would have gotten one earlier love it a...,19,5
1,neutral,i ve learned this lesson again open the packag...,88,1
2,neutral,it is so slow and lags find better option,9,2
3,neutral,roller ball stopped working within months of m...,12,1
4,neutral,i like the color and size but it few days out ...,21,1


In [3]:
# checking number of rows and columns
print("Shape of dataset:", df.shape)

# printing column names
print("\nColumns:")
print(df.columns)

# checking how many samples in each sentiment class
print("\nClass Distribution:")
print(df['sentiments'].value_counts())

Shape of dataset: (17340, 4)

Columns:
Index(['sentiments', 'cleaned_review', 'cleaned_review_length',
       'review_score'],
      dtype='object')

Class Distribution:
sentiments
positive    9503
neutral     6303
negative    1534
Name: count, dtype: int64


## Data Understanding

The dataset is loaded successfully and contains multiple columns.

The main feature used is "cleaned_review" and the target column is "sentiments".

The dataset has three classes: positive, negative, and neutral.

From the class distribution, we can see that ...

# 2. NLP Preprocessing

In [4]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# download stopwords (only once)
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

In [5]:
# loading stopwords
stop_words = set(stopwords.words('english'))

# using stemming
stemmer = PorterStemmer()

In [9]:
def preprocess_text(text):

    # handle missing values
    if pd.isna(text):
        return ""

    # convert to lowercase
    text = text.lower()

    # remove urls
    text = re.sub(r'http\S+', '', text)

    # remove special characters and numbers
    text = re.sub(r'[^a-z\s]', '', text)

    # tokenization
    words = text.split()

    # remove stopwords
    words = [w for w in words if w not in stop_words]

    # stemming
    words = [stemmer.stem(w) for w in words]

    return " ".join(words)

In [10]:
df['processed_text'] = df['cleaned_review'].apply(preprocess_text)

df[['cleaned_review', 'processed_text']].head()

,cleaned_review,processed_text
0,i wish would have gotten one earlier love it a...,wish would gotten one earlier love make work l...
1,i ve learned this lesson again open the packag...,learn lesson open packag use product right awa...
2,it is so slow and lags find better option,slow lag find better option
3,roller ball stopped working within months of m...,roller ball stop work within month minim use p...
4,i like the color and size but it few days out ...,like color size day return period hold charg


## NLP Preprocessing

In this step, the text data is cleaned and prepared for machine learning.

The following steps are applied:
- Converted text to lowercase
- Removed URLs and special characters
- Tokenized the text
- Removed stopwords
- Applied stemming

This helps in reducing noise and improving model performance.

# 3. Feature Engineering

In [11]:
# input features (text)
X = df['processed_text']

# target labels
y = df['sentiments']

In [14]:
# converting text into numerical features using Bag of Words
from sklearn.feature_extraction.text import CountVectorizer

# creating BoW model
bow = CountVectorizer()

# converting text into vectors
X_bow = bow.fit_transform(X)

print("BoW Shape:", X_bow.shape)

BoW Shape: (17340, 6421)


In [15]:
#converting text using TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

# creating TF-IDF model
tfidf = TfidfVectorizer()

# transforming text
X_tfidf = tfidf.fit_transform(X)

print("TF-IDF Shape:", X_tfidf.shape)

TF-IDF Shape: (17340, 6421)


## Feature Engineering

In this step, text data is converted into numerical form so that machine learning models can understand it.

Two techniques are used:

1. Bag of Words (BoW) – counts frequency of words  
2. TF-IDF – gives importance to important words and reduces the weight of common words  

TF-IDF generally performs better than BoW.

# 4. MODEL BUILDING

In [16]:
from sklearn.model_selection import train_test_split

# splitting data into train and test
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

In [20]:
# Logistic Regression is good for text classification
from sklearn.linear_model import LogisticRegression

# creating model
lr = LogisticRegression()

# training model
lr.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [21]:
# Naive Bayes works well with word frequencies
from sklearn.naive_bayes import MultinomialNB

# creating model
nb = MultinomialNB()

# training model
nb.fit(X_train, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [22]:
# Decision Tree is simple but may overfit
from sklearn.tree import DecisionTreeClassifier
# creating model
dt = DecisionTreeClassifier()
# training model
dt.fit(X_train, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


## Model Building

In this step, three machine learning models are trained:

- Logistic Regression
- Naive Bayes
- Decision Tree

These models are trained using TF-IDF features to classify text into sentiment categories.

# 5. Model Evaluation

In [23]:
from sklearn.metrics import accuracy_score, classification_report

In [26]:
# evaluating model performance using accuracy and classification report
# this helps compare which model performs better
# Logistic Regression evaluation
y_pred_lr = lr.predict(X_test)

print("Logistic Regression")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

# Naive Bayes evaluation
y_pred_nb = nb.predict(X_test)

print("\nNaive Bayes")
print("Accuracy:", accuracy_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb))

# Decision Tree evaluation
y_pred_dt = dt.predict(X_test)

print("\nDecision Tree")
print("Accuracy:", accuracy_score(y_test, y_pred_dt))
print(classification_report(y_test, y_pred_dt))

Logistic Regression
Accuracy: 0.8160322952710496
              precision    recall  f1-score   support

    negative       0.75      0.34      0.47       336
     neutral       0.73      0.82      0.78      1253
    positive       0.88      0.90      0.89      1879

    accuracy                           0.82      3468
   macro avg       0.79      0.69      0.71      3468
weighted avg       0.82      0.82      0.81      3468


Naive Bayes
Accuracy: 0.6871395617070357
              precision    recall  f1-score   support

    negative       0.50      0.00      0.01       336
     neutral       0.61      0.51      0.56      1253
    positive       0.72      0.93      0.81      1879

    accuracy                           0.69      3468
   macro avg       0.61      0.48      0.46      3468
weighted avg       0.66      0.69      0.64      3468


Decision Tree
Accuracy: 0.8370818915801614
              precision    recall  f1-score   support

    negative       0.73      0.65      0.69     

## Model Evaluation

The performance of all models is evaluated using accuracy, precision, recall, and F1-score.

- Logistic Regression performed the best overall.
- Naive Bayes showed good performance but slightly lower than Logistic Regression.
- Decision Tree had lower accuracy and may suffer from overfitting.

TF-IDF features helped improve model performance.

# 6. Comparison & Insights

## Comparison & Insights

- TF-IDF performed better than Bag of Words.
- Logistic Regression gave the highest accuracy.
- Naive Bayes was faster but slightly less accurate.
- Decision Tree was less stable and prone to overfitting.

Overall, Logistic Regression with TF-IDF is the best choice for this task.

## Conclusion

This project demonstrates how text data can be processed and used to build sentiment analysis models.

Different preprocessing techniques and feature engineering methods were applied. Multiple models were trained and compared.

Among all models, Logistic Regression performed best with TF-IDF features.